# Experimentul 12: evaluarea finală pe TEST a modelului STAR + Z₀Z₁

**Tip:** evaluare, nu antrenare. Nimic nu se reantrenează, nu se recalibrează și nu se selectează în acest notebook.

**Modele înghețate:**
- **A. STAR + Z₀Z₁** (candidatul): ponderile din `results/result_exp11/weights/Z0Z1`, alese prin regula de decizie din Exp. 11 (`decision.csv`), înainte de acest experiment.
- **B. STAR + Z₀** (referința cuantică): ponderile din `results/result_exp9/weights/star`.
- **C. Regresie logistică** pe aceleași 4 unghiuri: reconstruită determinist ca în Exp. 4 / 5 (coeficienții verificați).
- Istoric: baseline-ul A din Exp. 5 (VQC ZZ ×2), citit din `results/result_exp5` și nerulat din nou.

**Protocol:**
- `default.mixed`, valori de așteptare exacte (shots = None);
- `DepolarizingChannel(p)` după fiecare poartă nativă, pe fiecare qubit atins;
- p ∈ {0, 0.001, 0.01, 0.05};
- prag = max-F1 pe **validation** la același p (scorurile salvate în Exp. 11), aplicat neschimbat pe test: `pred = score >= prag`.

> ⚠️ **Setul de test nu este „blind”.** Testul a fost deja folosit în Exp. 5 (pentru modelele A și B ale acelui experiment), iar cercetarea a continuat după aceea. Evaluarea de față nu poate fi prezentată ca o verificare complet independentă de întregul proces exploratoriu.

**Seeds:** numpy / torch = 42 pentru verificări, `undersample(seed)` și `AngleEncoder(seed)` pentru seeds 0–4, iar bootstrap-ul folosește `default_rng(42)`.

## 1. Configurație

In [ ]:
import hashlib
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
from src.quantum_readout import (ANSATZ_REPS, N_QUBITS, NATIVE_GATES, OBSERVABLES, STAR_PAIRS, AngleEncoder,
                                 choose_threshold, evaluate, init_weights, make_model, make_vqc, noise_locations,
                                 noisy_predict, predict, set_seed, undersample)

# ---- modele înghețate: nimic nu se antrenează în Exp. 12 ----
CANDIDATE, QUANTUM_REF, LR_KEY, HIST_A = "Z0Z1", "Z0", "LR", "A"
QMODELS = [CANDIDATE, QUANTUM_REF]

# ---- protocolul oficial (identic cu Exp. 5 / 9 / 11) ----
SPLIT_SEED = 42
SEEDS = [0, 1, 2, 3, 4]
NOISE_LEVELS = [0.001, 0.01, 0.05]
P_ALL = [0.0, *NOISE_LEVELS]
P_FOCUS = [0.01, 0.05]
LR_MAX_ITER = 1000
N_TEST_EXPECTED, N_TEST_FRAUDS_EXPECTED = 56962, 98

# ---- cost și analize descriptive ----
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger-ul nominal; metricile oficiale sunt exacte (shots = None)
N_BOOTSTRAP = 1000            # bootstrap stratificat pereche pe test: doar incertitudine descriptivă, după evaluare
BOOTSTRAP_SEED = SPLIT_SEED

# ---- verificări ----
SPOT_CHECK_ROWS = 2000        # rânduri pentru recalcularea scorurilor salvate (validation Exp. 11, test Exp. 5)
EXACT_TOL = 1e-12
NOISY_TOL = 1e-10
FULL_PAIRS = [(i, j) for i in range(N_QUBITS) for j in range(i + 1, N_QUBITS)]   # ZZ complet: doar modelul B din Exp. 5

RESULTS_DIR = REPO_ROOT / "results" / "result_exp12"
TEST_SCORES_DIR = RESULTS_DIR / "test_scores"
PLOTS_DIR = RESULTS_DIR / "plots"
EXP3_DIR, EXP4_DIR, EXP5_DIR, EXP9_DIR, EXP11_DIR = (REPO_ROOT / "results" / f"result_exp{k}" for k in [3, 4, 5, 9, 11])   # doar citite
SPLIT_PATH = REPO_ROOT / "split_indices.npz"
WEIGHT_PATHS = {CANDIDATE: lambda s: EXP11_DIR / "weights" / "Z0Z1" / f"seed_{s}.pt",
                QUANTUM_REF: lambda s: EXP9_DIR / "weights" / "star" / f"seed_{s}.pt"}

LABELS = {CANDIDATE: "STAR + Z₀Z₁ (candidat, Exp. 11)", QUANTUM_REF: "STAR + Z₀ (Exp. 9)",
          LR_KEY: "regresie logistică (Exp. 4)", HIST_A: "A: VQC ZZ ×2 (Exp. 5, istoric)"}
SHORT = {CANDIDATE: "Z₀Z₁", QUANTUM_REF: "Z₀", LR_KEY: "LR", HIST_A: "A (Exp. 5)"}
COLORS = {CANDIDATE: "#e87ba4", QUANTUM_REF: "#2a78d6", LR_KEY: "#52514e", HIST_A: "#eda100"}   # aceleași culori ca în Exp. 11
MARKERS = {CANDIDATE: "P", QUANTUM_REF: "o", LR_KEY: "s", HIST_A: "D"}
SEED_DOT_COLOR = "#8a8f98"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

torch.set_default_dtype(torch.float64)
set_seed(SPLIT_SEED)
for d in [RESULTS_DIR, TEST_SCORES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())


def load_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


checks = []


def check(name, ok, n=None, err=None):
    assert ok, f"verificare eșuată: {name}"
    checks.append({"verificare": name, "intrări": n, "trecut": True, "eroare max": err})


print(f"modele: {QMODELS} + {LR_KEY} (+ {HIST_A} istoric)   seeds: {SEEDS}   p: {P_ALL}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| numpy", np.__version__)

## 2. Datele și splitul

Splitul se reconstruiește cu seed 42 și se compară cu `split_indices.npz`, apoi verificăm hash-urile salvate în Exp. 5 și Exp. 11. Cele două experimente au folosit convenții diferite de hash, așa că le verificăm pe amândouă.

Etichetele test se încarcă doar pentru a verifica dimensiunile (56 962 / 98). Rămân în `_Y_TEST_LOCKED` și se folosesc abia în §6, după ce pragurile sunt înghețate și salvate.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
for path in [DATA_PATH, SPLIT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and df.index.equals(pd.RangeIndex(len(df)))

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
check("split_indices.npz = reconstrucția oficială (60/20/20 stratificat, seed 42)",
      np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), 3)
check("fără suprapuneri între train, validation și test",
      not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx)), 3)
check("acoperire completă: train + val + test = 284 807 de tranzacții distincte",
      len(np.unique(np.r_[train_idx, val_idx, test_idx])) == len(df), len(df))

SPLIT_SHA256_ARRAYS = sha256_bytes(b"".join(np.ascontiguousarray(split[k]).tobytes() for k in ["train", "val", "test"]))
SPLIT_SHA256_FILE = sha256_file(SPLIT_PATH)
exp11_config = json.loads((EXP11_DIR / "config.json").read_text(encoding="utf-8"))
exp5_config = json.loads((EXP5_DIR / "config.json").read_text(encoding="utf-8"))
check("SHA-256 split = Exp. 11 (convenția: train + val + test concatenate)", SPLIT_SHA256_ARRAYS == exp11_config["split_sha256"], 1)
check("SHA-256 split = Exp. 5 (convenția: fișierul split_indices.npz)", SPLIT_SHA256_FILE == exp5_config["split_sha256"], 1)

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
X_test = X_all.loc[test_idx]
_Y_TEST_LOCKED = y_all[test_idx]   # folosit doar pentru dimensiuni până în §6
check("dimensiuni: train 170 883 / 295, validation 56 962 / 99, test 56 962 / 98",
      (len(y_train), int(y_train.sum()), len(y_val), int(y_val.sum()), len(_Y_TEST_LOCKED), int(_Y_TEST_LOCKED.sum()))
      == (170883, 295, 56962, 99, N_TEST_EXPECTED, N_TEST_FRAUDS_EXPECTED), 3)
check("ordinea etichetelor: X și y în ordinea indicilor din split", X_test.index.equals(pd.Index(test_idx)) and X_val.index.equals(pd.Index(val_idx)), 2)
del X_all, df, _trval, _test, _train, _val
print(f"train {len(y_train):,} ({int(y_train.sum())})   validation {len(y_val):,} ({int(y_val.sum())})   "
      f"test {len(_Y_TEST_LOCKED):,} ({int(_Y_TEST_LOCKED.sum())})")
print("SHA-256 split:", SPLIT_SHA256_ARRAYS[:16], "… (Exp. 11) /", SPLIT_SHA256_FILE[:16], "… (Exp. 5)")

## 3. Circuitele: identice cu Exp. 11

In [ ]:
CIRCUITS = {o: make_vqc(STAR_PAIRS, o) for o in QMODELS}
dev4 = qml.device("default.qubit", wires=N_QUBITS)   # shots = None: valori de așteptare exacte
QNODES = {o: qml.QNode(CIRCUITS[o], dev4, interface="torch", diff_method="backprop") for o in QMODELS}
MODELS = {o: make_model(QNODES[o]) for o in QMODELS}   # s(x) = (1 - <O>) / 2


def tape_of(circuit_fn, x, w):
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, dev4), gate_set=NATIVE_GATES)
    return qml.workflow.construct_tape(qnode)(x, w)


rng_check = np.random.default_rng(SPLIT_SEED)
X_probe = rng_check.uniform(0, np.pi, (10, N_QUBITS))
W_probe = rng_check.uniform(0, 2 * np.pi, (10, ANSATZ_REPS + 1, N_QUBITS, 2))
expected_obs = {"Z0": qml.PauliZ(0), "Z0Z1": qml.PauliZ(0) @ qml.PauliZ(1)}
for x, w in zip(X_probe, W_probe):
    for o in QMODELS:
        tape = tape_of(CIRCUITS[o], x, w)
        assert [f"{op.name}{list(op.wires)}" for op in tape.operations] == exp11_config["native_op_order"], o
        assert len(tape.measurements) == 1 and qml.equal(tape.measurements[0].obs, expected_obs[o])
check("ordinea porților native = Exp. 11 (config.json), pentru Z0 și Z0Z1", True, len(X_probe) * 2)
check("observabila măsurată: Z0 = PauliZ(0), Z0Z1 = PauliZ(0) @ PauliZ(1) (ca în Exp. 11)",
      all(exp11_config["observables"][o] == {"Z0": "qml.PauliZ(0)", "Z0Z1": "qml.PauliZ(0) @ qml.PauliZ(1)"}[o] for o in QMODELS), 2)

STRUCT = {}
x_t, w_t = torch.tensor(X_probe[0]), torch.tensor(W_probe[0])
for o in QMODELS:
    ops = tape_of(CIRCUITS[o], X_probe[0], W_probe[0]).operations
    STRUCT[o] = {"qubiți": N_QUBITS, "porți native": len(ops), "CNOT": sum(op.name == "CNOT" for op in ops),
                 "locuri de zgomot": noise_locations(CIRCUITS[o], x_t, w_t), "parametri": (ANSATZ_REPS + 1) * N_QUBITS * 2}
    assert (STRUCT[o]["CNOT"], STRUCT[o]["locuri de zgomot"], STRUCT[o]["parametri"]) == (15, 73, 32), STRUCT[o]
check("4 qubiți, 15 CNOT, 73 locuri de zgomot, 32 parametri (ambele observabile)", True, 2)
display(pd.DataFrame(STRUCT).T)

## 4. Encodere, ponderi și regresia logistică: refolosite, fără reantrenare

Encoderul fiecărui seed este `AngleEncoder(4, seed)` (StandardScaler pe Time / Amount → PCA(4) → MinMaxScaler în [0, π] + clipping). Este fitat **doar** pe cele 590 de exemple 1:1 din train, exact ca în Exp. 9 / 11. Encoderele nu au fost salvate ca obiecte, dar reconstrucția este deterministă. Dovada: unghiurile train + val au același SHA-256 ca în Exp. 9 și Exp. 11.

Pe test se aplică doar `transform`. Verificăm că parametrii encoderului nu se schimbă după transformarea testului.

In [ ]:
def encoder_sha256(enc):
    parts = [enc.scaler.mean_, enc.scaler.scale_, enc.pca.mean_, enc.pca.components_,
             enc.angles.data_min_, enc.angles.data_max_, enc.angles.scale_, enc.angles.min_]
    return sha256_bytes(b"".join(np.ascontiguousarray(a, dtype=np.float64).tobytes() for a in parts))


exp9_train = {r["seed"]: r for r in load_jsonl(EXP9_DIR / "train_runs.jsonl") if r["topologie"] == "star"}
exp11_train = {(r["observabilă"], r["seed"]): r for r in load_jsonl(EXP11_DIR / "train_runs.jsonl")}
exp11_metrics = pd.read_csv(EXP11_DIR / "metrics_per_seed.csv")
exp4_val = {r["seed"]: r for r in load_jsonl(EXP4_DIR / "val_runs.jsonl")}


def exp11_row(o, seed, p):
    return exp11_metrics[(exp11_metrics["observabilă"] == o) & (exp11_metrics["seed"] == seed) & (exp11_metrics["p"] == p)].iloc[0]


data, weights, weight_sha, lr_models, model_rows = {}, {o: {} for o in QMODELS}, {}, {}, []
test_set = set(test_idx)
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    assert len(y_s) == 590 and int(y_s.sum()) == 295 and set(X_s.index) <= set(train_idx) and not (set(X_s.index) & test_set)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)   # fit doar pe train (590 exemple)
    assert encoder.scaler.n_samples_seen_ == 590 and encoder.pca.n_samples_ == 590 and encoder.angles.n_samples_seen_ == 590
    Z_tr, Z_val = encoder.transform(X_s), encoder.transform(X_val)
    z_sha = sha256_bytes(Z_tr.tobytes() + Z_val.tobytes())
    assert z_sha == exp9_train[seed]["sha256 unghiuri (train + val)"] == exp11_train[(CANDIDATE, seed)]["sha256 unghiuri (train + val)"], seed
    enc_sha = encoder_sha256(encoder)
    Z_test = encoder.transform(X_test)                # doar transform: niciun fit pe test
    assert encoder_sha256(encoder) == enc_sha and Z_test.shape == (N_TEST_EXPECTED, N_QUBITS)
    assert Z_test.min() >= 0 and Z_test.max() <= np.pi
    data[seed] = {"Z_tr": Z_tr, "y_tr": y_s, "Z_val": Z_val, "Z_test": Z_test, "encoder_sha256": enc_sha,
                  "angles_train_val_sha256": z_sha, "angles_test_sha256": sha256_bytes(Z_test.tobytes()),
                  "test_clipped_fraction": float(np.mean((Z_test == 0) | (Z_test == np.pi)))}

    for o in QMODELS:
        path = WEIGHT_PATHS[o](seed)
        saved = torch.load(path, weights_only=True)
        assert saved["weights"].shape == (ANSATZ_REPS + 1, N_QUBITS, 2) and saved["weights"].dtype == torch.float64
        assert torch.equal(saved["initial_weights"], init_weights(seed).detach()), f"{o} seed {seed}: altă inițializare"
        clean_val = predict(Z_val, saved["weights"], MODELS[o])
        ap = average_precision_score(y_val, clean_val)
        ref = exp11_row(o, seed, 0.0)
        ref_epoch = exp9_train[seed]["best_epoch"] if o == QUANTUM_REF else exp11_train[(o, seed)]["best_epoch"]
        assert abs(ap - saved["best_val_ap"]) < EXACT_TOL and abs(ap - ref["AUPRC"]) < EXACT_TOL, (o, seed, ap)
        assert saved["best_epoch"] == ref_epoch == ref["epoca aleasă"], (o, seed)
        weights[o][seed] = saved["weights"]
        weight_sha[f"{o}/seed_{seed}"] = sha256_file(path)
        data[seed][f"clean_val_{o}"] = clean_val
        model_rows.append({"model": o, "seed": seed, "fișier": str(path.relative_to(REPO_ROOT)), "epoca aleasă": saved["best_epoch"],
                           "val AUPRC p = 0 recalculat": ap, "val AUPRC Exp. 11": ref["AUPRC"], "sha256": weight_sha[f"{o}/seed_{seed}"][:16]})

    set_seed(seed)
    lr = LogisticRegression(max_iter=LR_MAX_ITER).fit(Z_tr, y_s)   # deterministă (lbfgs), doar pe train
    ref = exp4_val[seed]
    assert np.allclose(lr.coef_[0], ref["coeficienți"], rtol=0, atol=1e-10) and abs(lr.intercept_[0] - ref["intercept"]) < 1e-10
    lr_models[seed] = lr

check("subseturi 1:1 ⊂ train, disjuncte de test; encoder fitat doar pe 590 de exemple de train", True, len(SEEDS))
check("unghiuri train + val identice cu Exp. 9 și Exp. 11 (SHA-256) ⇒ același encoder per seed", True, len(SEEDS))
check("encoderul nu se modifică la transformarea testului (hash parametri înainte = după)", True, len(SEEDS))
check("ponderi Z0 (Exp. 9) și Z0Z1 (Exp. 11): inițializare, epocă, val AUPRC p = 0 reproduse exact", True, len(SEEDS) * 2)
check("regresia logistică reconstruită: coeficienți și intercept = Exp. 4", True, len(SEEDS))
display(pd.DataFrame(model_rows).set_index(["model", "seed"]))
print("fracția unghiurilor test atinse de clipping [0, π]:", {s: round(data[s]["test_clipped_fraction"], 5) for s in SEEDS})

## 5. Praguri înghețate, calculate doar din validation

- **VQC:** pragurile vin din scorurile validation salvate în Exp. 11 (`result_exp11/scores`), cu aceeași regulă `choose_threshold` (max-F1).
  - La fiecare (model, seed, p) verificăm că pragul este identic cu cel din `metrics_per_seed.csv`.
  - Recalculăm independent primele 2 000 de scoruri validation din ponderi, pentru toate nivelurile de zgomot. Asta verifică și că modelul de zgomot de aici este cel din Exp. 11.
- **LR:** pragul max-F1 pe validation, verificat față de Exp. 4.

Pragurile se salvează în `thresholds.csv`, iar hash-ul lor intră în `config.json` **înainte** de calculul scorurilor test.

In [ ]:
VAL_SCORES, THRESHOLDS, thr_rows = {}, {}, []
for o in QMODELS:
    for seed in SEEDS:
        for p in P_ALL:
            s = np.load(EXP11_DIR / "scores" / f"{o}_seed{seed}_p{p}.npz")["val"]
            assert s.shape == y_val.shape and np.isfinite(s).all()
            if p == 0.0:
                assert np.abs(s - data[seed][f"clean_val_{o}"]).max() < EXACT_TOL
            thr, ref = choose_threshold(y_val, s), exp11_row(o, seed, p)
            assert abs(thr - ref["threshold"]) < EXACT_TOL and abs(average_precision_score(y_val, s) - ref["AUPRC"]) < EXACT_TOL, (o, seed, p)
            VAL_SCORES[o, seed, p], THRESHOLDS[o, seed, p] = s, float(thr)
            thr_rows.append({"model": o, "seed": seed, "p": p, "threshold": float(thr), "sursa": f"result_exp11/scores/{o}_seed{seed}_p{p}.npz (val)"})
for seed in SEEDS:
    s = lr_models[seed].predict_proba(data[seed]["Z_val"])[:, 1]
    thr = choose_threshold(y_val, s)
    assert abs(thr - exp4_val[seed]["threshold"]) < 1e-12
    for p in P_ALL:
        VAL_SCORES[LR_KEY, seed, p], THRESHOLDS[LR_KEY, seed, p] = s, float(thr)
        thr_rows.append({"model": LR_KEY, "seed": seed, "p": p, "threshold": float(thr), "sursa": "LR reconstruită, validation (= Exp. 4)"})
check("praguri VQC = max-F1 pe scorurile validation din Exp. 11 (identice cu metrics_per_seed.csv)", True, len(QMODELS) * len(SEEDS) * len(P_ALL), 0.0)
check("prag LR = Exp. 4", True, len(SEEDS))

spot_err = 0.0
spot_start = time.perf_counter()
for o in QMODELS:
    for seed in SEEDS:
        for p in NOISE_LEVELS:
            s_new = noisy_predict(data[seed]["Z_val"][:SPOT_CHECK_ROWS], weights[o][seed], p, CIRCUITS[o])
            spot_err = max(spot_err, np.abs(s_new - VAL_SCORES[o, seed, p][:SPOT_CHECK_ROWS]).max())
SPOT_SECONDS = time.perf_counter() - spot_start
check(f"modelul de zgomot = Exp. 11: scoruri validation noisy recalculate ({SPOT_CHECK_ROWS} rânduri × 2 modele × 5 seeds × 3 p)",
      spot_err < NOISY_TOL, len(QMODELS) * len(SEEDS) * len(NOISE_LEVELS), spot_err)

thresholds = pd.DataFrame(thr_rows)
THRESHOLDS_PATH = RESULTS_DIR / "thresholds.csv"
THRESHOLDS_SHA256 = sha256_bytes(np.array([THRESHOLDS[k] for k in sorted(THRESHOLDS, key=str)]).tobytes())
if THRESHOLDS_PATH.exists():
    previous = pd.read_csv(THRESHOLDS_PATH)
    assert np.allclose(previous["threshold"], thresholds["threshold"], rtol=0, atol=0), "thresholds.csv existent diferă: nu suprascriu"
else:
    thresholds.to_csv(THRESHOLDS_PATH, index=False)

config = {
    "experiment": "final_test_evaluation_exp12",
    "type": "evaluation only: no training, no recalibration, no model selection",
    "models": {
        CANDIDATE: {"label": LABELS[CANDIDATE], "observable": "qml.PauliZ(0) @ qml.PauliZ(1)", "weights": "results/result_exp11/weights/Z0Z1",
                    "selected_by": "Exp. 11 decision rule (results/result_exp11/decision.csv), on validation, before Exp. 12"},
        QUANTUM_REF: {"label": LABELS[QUANTUM_REF], "observable": "qml.PauliZ(0)", "weights": "results/result_exp9/weights/star"},
        LR_KEY: {"label": LABELS[LR_KEY], "config": f"LogisticRegression(max_iter={LR_MAX_ITER}) on the same 4 angles; coefficients verified against Exp. 4"},
        HIST_A: {"label": LABELS[HIST_A], "source": "results/result_exp5/test_detailed.csv (historical, not re-run)"},
    },
    "circuit": {"pairs": STAR_PAIRS, "ansatz_reps": ANSATZ_REPS, **STRUCT[CANDIDATE], "native_op_order": "= results/result_exp11/config.json"},
    "score": "s(x) = (1 - <O>) / 2",
    "seeds": SEEDS, "noise_levels": P_ALL,
    "noise_model": "noisy_qnode: NATIVE_GATES decomposition, DepolarizingChannel(p) after every gate on every wire, default.mixed, exact expectations",
    "threshold_rule": "max F1 on validation at the same p (Exp. 11 saved validation scores); test prediction = score >= threshold",
    "thresholds_sha256": THRESHOLDS_SHA256, "thresholds_frozen_before_test_scoring": True,
    "encoder": "AngleEncoder(4, seed) fitted on undersample(X_train, y_train, seed) 1:1 (590 rows); transform only on test",
    "encoder_params_sha256": {s: data[s]["encoder_sha256"] for s in SEEDS},
    "angles_train_val_sha256": {s: data[s]["angles_train_val_sha256"] for s in SEEDS},
    "angles_test_sha256": {s: data[s]["angles_test_sha256"] for s in SEEDS},
    "weights_sha256": weight_sha,
    "split": "split_indices.npz (verified against official reconstruction)",
    "split_sha256_arrays_exp11_convention": SPLIT_SHA256_ARRAYS, "split_sha256_file_exp5_convention": SPLIT_SHA256_FILE,
    "test_prior_use": "The test set was already used in Exp. 5 (models A = Exp. 2 and B = Exp. 3). This evaluation is NOT fully blind / independent of the exploratory process.",
    "n_test": N_TEST_EXPECTED, "n_test_frauds": N_TEST_FRAUDS_EXPECTED, "shots_per_execution_nominal": SHOTS_PER_EXECUTION,
    "bootstrap": {"n": N_BOOTSTRAP, "seed": BOOTSTRAP_SEED, "type": "stratified paired over test transactions, descriptive only"},
    "torch_default_dtype": str(torch.get_default_dtype()),
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
CONFIG_PATH = RESULTS_DIR / "config.json"
current = json.loads(json.dumps(to_builtin(config)))
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu.")
else:
    CONFIG_PATH.write_text(json.dumps(current, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✅ praguri înghețate ({len(THRESHOLDS)}), sha256 {THRESHOLDS_SHA256[:16]}…; recalculare val noisy: eroare max {spot_err:.1e}")
display(thresholds.pivot_table(index=["model", "seed"], columns="p", values="threshold").round(4))

### 5b. Verificarea #7: reproducem Exp. 5 cu același pipeline

**STAR + Z₀ nu a fost evaluat pe test în Exp. 5.** Exp. 5 a testat A (Exp. 2, ZZ ×2) și B (Exp. 3, ZZ ×1, toate perechile), iar STAR a apărut abia în Exp. 9. Verificarea „Z₀ pe test reproduce Exp. 5” nu se poate face literal.

Facem în schimb verificarea echivalentă a pipeline-ului: același encoder, aceeași funcție `make_vqc` (cu toate perechile ZZ), același `noisy_predict` și ponderile B din Exp. 3. Rezultatul trebuie să reproducă scorurile test salvate de Exp. 5:
- p = 0: toate cele 56 962 de tranzacții;
- p > 0: primele 2 000.

LR trebuie să reproducă exact scorurile test ale modelului C din Exp. 5. Comparăm doar scoruri, deci etichetele test rămân blocate.

**Tipul implicit din torch.** Exp. 5 nu a setat `torch.set_default_dtype`, deci a rulat cu float32 implicit. Cu acea setare, `default.mixed` dă scoruri care diferă de cele obținute cu float64 (Exp. 9 / 11 / 12) cu ~1e-9.
- Verificarea strictă rulează cu setarea Exp. 5, apoi revenim la float64.
- Diferența float64 vs Exp. 5 este raportată separat, informativ: ea nu schimbă clasamentul.

In [ ]:
B_CIRCUIT = make_vqc(FULL_PAIRS, "Z0")
B_MODEL = make_model(qml.QNode(B_CIRCUIT, dev4, interface="torch", diff_method="backprop"))
exp5_struct = pd.read_csv(EXP5_DIR / "circuit_structure.csv", index_col=0)
b_ops = tape_of(B_CIRCUIT, X_probe[0], W_probe[0]).operations
assert (sum(op.name == "CNOT" for op in b_ops), noise_locations(B_CIRCUIT, x_t, w_t)) == (int(exp5_struct.iloc[1]["CNOT"]), int(exp5_struct.iloc[1]["locuri de zgomot"]))

b_err0 = lr_err = 0.0
b_err_noisy = {torch.float32: 0.0, torch.float64: 0.0}   # tipul implicit torch: Exp. 5 (float32) / Exp. 9-12 (float64)
for seed in SEEDS:
    path = EXP3_DIR / "weights" / f"seed_{seed}.pt"
    assert sha256_file(path) == exp5_config["weights_sha256"][f"B/seed_{seed}.pt"]
    w_b = torch.load(path, weights_only=True)["weights"]
    b_err0 = max(b_err0, np.abs(predict(data[seed]["Z_test"], w_b, B_MODEL) - np.load(EXP5_DIR / "scores" / f"B_seed{seed}_p0.0.npz")["test"]).max())
    for p in NOISE_LEVELS:
        saved = np.load(EXP5_DIR / "scores" / f"B_seed{seed}_p{p}.npz")["test"][:SPOT_CHECK_ROWS]
        for dtype in b_err_noisy:
            torch.set_default_dtype(dtype)
            try:
                s_b = noisy_predict(data[seed]["Z_test"][:SPOT_CHECK_ROWS], w_b, p, B_CIRCUIT)
            finally:
                torch.set_default_dtype(torch.float64)
            b_err_noisy[dtype] = max(b_err_noisy[dtype], np.abs(s_b - saved).max())
    lr_err = max(lr_err, np.abs(lr_models[seed].predict_proba(data[seed]["Z_test"])[:, 1] - np.load(EXP5_DIR / "scores" / f"C_seed{seed}_p0.0.npz")["test"]).max())
assert torch.get_default_dtype() == torch.float64
check("#7 (echivalent) pipeline-ul reproduce scorurile test Exp. 5, model B, p = 0 (56 962 tranzacții × 5 seeds)", b_err0 < 1e-10, len(SEEDS), b_err0)
check(f"#7 (echivalent) pipeline-ul reproduce scorurile test Exp. 5, model B, p = 0.001 / 0.01 / 0.05 ({SPOT_CHECK_ROWS} rânduri), "
      "cu tipul implicit torch al Exp. 5 (float32)", b_err_noisy[torch.float32] < NOISY_TOL, len(SEEDS) * 3, b_err_noisy[torch.float32])
checks.append({"verificare": "informativ (nu e criteriu): același calcul cu float64 implicit (protocolul Exp. 9 / 11 / 12), diferența max față de scorurile Exp. 5",
               "intrări": len(SEEDS) * 3, "trecut": None, "eroare max": b_err_noisy[torch.float64]})
check("LR reconstruită reproduce scorurile test Exp. 5 (model C)", lr_err < EXACT_TOL, len(SEEDS), lr_err)
print(f"✅ Exp. 5 reprodus: B p = 0 eroare {b_err0:.1e}, B noisy {b_err_noisy[torch.float32]:.1e} (float32 ca Exp. 5; "
      f"float64: {b_err_noisy[torch.float64]:.1e}), LR {lr_err:.1e}")

## 6. Scorurile test (exacte, `default.mixed`)

Un scor per (model, seed, p) pentru toate cele 56 962 de tranzacții. Scorurile se salvează în `test_scores/`. La o rulare repetată se reîncarcă, nu se suprascriu.

In [ ]:
def test_scores_path(model, seed, p):
    return TEST_SCORES_DIR / f"{model}_seed{seed}_p{p}.npz"


TEST_SCORES, TEST_TIME = {}, {}
todo = [(o, s, p) for o in QMODELS for s in SEEDS for p in P_ALL]
print(f"de calculat: {sum(not test_scores_path(*t).exists() for t in todo)} din {len(todo)}")
for o, seed, p in todo:
    path = test_scores_path(o, seed, p)
    if not path.exists():
        start = time.perf_counter()
        Z_t, w = data[seed]["Z_test"], weights[o][seed]
        s = predict(Z_t, w, MODELS[o]) if p == 0.0 else noisy_predict(Z_t, w, p, CIRCUITS[o])
        seconds = time.perf_counter() - start
        tmp = path.with_name(path.stem + ".tmp.npz")
        np.savez_compressed(tmp, test=s, seconds=seconds, weights_sha256=weight_sha[f"{o}/seed_{seed}"], threshold_from_validation=THRESHOLDS[o, seed, p])
        tmp.replace(path)
        print(f"{o:<5} seed {seed}  p = {p:<5}  calculat ({seconds:.0f} s)")
    z = np.load(path)
    assert str(z["weights_sha256"]) == weight_sha[f"{o}/seed_{seed}"] and float(z["threshold_from_validation"]) == THRESHOLDS[o, seed, p]
    TEST_SCORES[o, seed, p], TEST_TIME[o, seed, p] = z["test"], float(z["seconds"])
    assert TEST_SCORES[o, seed, p].shape == (N_TEST_EXPECTED,) and np.isfinite(TEST_SCORES[o, seed, p]).all()
    assert TEST_SCORES[o, seed, p].min() >= -EXACT_TOL and TEST_SCORES[o, seed, p].max() <= 1 + EXACT_TOL
for seed in SEEDS:
    path = TEST_SCORES_DIR / f"{LR_KEY}_seed{seed}.npz"
    start = time.perf_counter()
    s = lr_models[seed].predict_proba(data[seed]["Z_test"])[:, 1]
    seconds = time.perf_counter() - start
    if not path.exists():
        np.savez_compressed(path, test=s, seconds=seconds)
    assert np.array_equal(np.load(path)["test"], s)
    for p in P_ALL:
        TEST_SCORES[LR_KEY, seed, p], TEST_TIME[LR_KEY, seed, p] = s, seconds

mixed_err = max(np.abs(noisy_predict(data[s]["Z_test"][:SPOT_CHECK_ROWS], weights[o][s], 0.0, CIRCUITS[o]) - TEST_SCORES[o, s, 0.0][:SPOT_CHECK_ROWS]).max()
                for o in QMODELS for s in SEEDS)
check("scoruri test complete (2 modele × 5 seeds × 4 p), finite, în [0, 1]", True, len(todo))
check(f"default.mixed la p = 0 = default.qubit pe test ({SPOT_CHECK_ROWS} rânduri)", mixed_err < 1e-10, len(QMODELS) * len(SEEDS), mixed_err)
print(f"✅ scoruri test gata; timp total de simulare (salvat): {sum(TEST_TIME[t] for t in todo) / 60:.1f} min")

## 7. Metrici pe test

Etichetele test se deblochează abia aici. Pragurile sunt deja înghețate (`thresholds.csv`), iar înainte de a le aplica verificăm din nou hash-ul lor.

In [ ]:
assert sha256_bytes(np.array([THRESHOLDS[k] for k in sorted(THRESHOLDS, key=str)]).tobytes()) == json.loads(CONFIG_PATH.read_text(encoding="utf-8"))["thresholds_sha256"]
y_test = _Y_TEST_LOCKED   # deblocat după înghețarea pragurilor
N_LEGIT_TEST, N_LEGIT_VAL = int((y_test == 0).sum()), int((y_val == 0).sum())

rows = []
for o in [*QMODELS, LR_KEY]:
    for seed in SEEDS:
        clean = TEST_SCORES[o, seed, 0.0]
        ap0 = average_precision_score(y_test, clean)
        for p in P_ALL:
            s, thr = TEST_SCORES[o, seed, p], THRESHOLDS[o, seed, p]
            m = evaluate(y_test, s, thr)
            is_q = o != LR_KEY
            diff = s - clean
            rows.append({"model": o, "seed": seed, "p": p, **m, "threshold (validation)": thr,
                         "Spearman rho": spearmanr(clean, s).statistic if is_q else np.nan,
                         "MAE scor": float(np.abs(diff).mean()) if is_q else np.nan,
                         "RMSE scor": float(np.sqrt(np.mean(diff ** 2))) if is_q else np.nan,
                         "AUPRC_drop (vs p = 0)": ap0 - m["AUPRC"], "FP / 10k legitime": 1e4 * m["FP"] / N_LEGIT_TEST,
                         "scoruri unice": len(np.unique(s)), "timp evaluare [s]": TEST_TIME[o, seed, p]})
test_metrics = pd.DataFrame(rows)
for c in ["TP", "FP", "FN", "TN"]:
    test_metrics[c] = test_metrics[c].astype(int)

ok_cm = ((test_metrics["TP"] + test_metrics["FN"] == 98) & (test_metrics["FP"] + test_metrics["TN"] == 56864)
         & (test_metrics[["TP", "FP", "FN", "TN"]].sum(axis=1) == 56962)).all()
check("matrice de confuzie: TP + FN = 98, FP + TN = 56 864, total = 56 962 (toate rândurile)", ok_cm, len(test_metrics))
q0 = test_metrics[(test_metrics["model"] != LR_KEY) & (test_metrics["p"] == 0.0)]
check("Spearman ρ la p = 0 față de propriile scoruri ideale = 1", np.allclose(q0["Spearman rho"], 1.0, atol=1e-12), len(q0))
check("AUPRC din scoruri continue (average_precision_score pe scoruri, ≥ 1 000 de valori unice)", (test_metrics["scoruri unice"] >= 1000).all(), len(test_metrics))

# sanity: aceeași funcție evaluate dă exact metricile istorice ale Exp. 5 din scorurile lui salvate
exp5_detail = pd.read_csv(EXP5_DIR / "test_detailed.csv")
e5_err = 0.0
for _, r in exp5_detail.iterrows():
    z = np.load(EXP5_DIR / "scores" / f"{r['model']}_seed{int(r['seed'])}_p{r['p']}.npz")
    m = evaluate(y_test, z["test"], r["threshold"])
    e5_err = max(e5_err, abs(m["AUPRC"] - r["AUPRC"]), abs(m["TP"] - r["TP"]), abs(m["FP"] - r["FP"]))
lr_vs_e5 = max(abs(test_metrics[(test_metrics["model"] == LR_KEY) & (test_metrics["p"] == 0.0)].set_index("seed").loc[s, k]
                   - exp5_detail[(exp5_detail["model"] == "C") & (exp5_detail["seed"] == s)].iloc[0][k]) for s in SEEDS for k in ["AUPRC", "TP", "FP", "threshold"]
               if k != "threshold")
check("metricile istorice Exp. 5 (A, B, C) recalculate din scorurile salvate = test_detailed.csv", e5_err < 1e-12, len(exp5_detail), e5_err)
check("LR pe test = modelul C din Exp. 5 (AUPRC, TP, FP)", lr_vs_e5 < 1e-12, len(SEEDS), lr_vs_e5)
test_metrics.to_csv(RESULTS_DIR / "test_metrics_per_seed.csv", index=False)

# tabel comun, cu baseline-ul A din Exp. 5 (istoric, aceleași seeds și același protocol)
hist_a = exp5_detail[exp5_detail["model"] == "A"].rename(columns={"threshold": "threshold (validation)"}).assign(model=HIST_A)
hist_a["FP / 10k legitime"] = 1e4 * hist_a["FP"] / N_LEGIT_TEST
ALL = pd.concat([test_metrics, hist_a[[c for c in hist_a.columns if c in test_metrics.columns]]], ignore_index=True)
ALL_MODELS = [CANDIDATE, QUANTUM_REF, LR_KEY, HIST_A]


def per_seed(model, p, metric):
    return ALL[(ALL["model"] == model) & (ALL["p"] == p)].set_index("seed").loc[SEEDS, metric].astype(float)


SUMMARY_METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN",
                   "FP / 10k legitime", "MAE scor", "RMSE scor", "AUPRC_drop (vs p = 0)", "threshold (validation)", "timp evaluare [s]"]
summary_rows = []
for model in ALL_MODELS:
    for p in P_ALL:
        for m in SUMMARY_METRICS:
            if m not in ALL.columns:
                continue
            v = per_seed(model, p, m)
            if v.isna().all():
                continue
            summary_rows.append({"model": model, "p": p, "metrică": m, "medie": v.mean(), "std": v.std(ddof=1), "min": v.min(), "max": v.max(),
                                 **{f"seed {s}": v[s] for s in SEEDS}})
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS_DIR / "test_summary_metrics.csv", index=False)

view = pd.DataFrame([{"model": SHORT[mo], "p": p, **{m: mean_std(per_seed(mo, p, m), 1 if m in ("TP", "FP") else 3)
                      for m in ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP"]},
                      "Spearman ρ": mean_std(per_seed(mo, p, "Spearman rho"), 4) if mo in QMODELS + [HIST_A] else "—"}
                     for mo in ALL_MODELS for p in P_ALL]).set_index(["model", "p"])
print("TEST, medie ± std pe 5 seeds (prag max-F1 din validation la același p)")
display(view)

## 8. Comparații pereche (AUPRC și metricile la prag)

In [ ]:
COMPARISONS = [("A: Z0Z1 vs Z0", CANDIDATE, QUANTUM_REF), ("B: Z0Z1 vs LR", CANDIDATE, LR_KEY),
               ("C: Z0 vs LR", QUANTUM_REF, LR_KEY), ("D: Z0Z1 vs A (Exp. 5)", CANDIDATE, HIST_A)]
PAIR_METRICS = ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP"]
paired_rows = []
for name, first, second in COMPARISONS:
    for p in P_ALL:
        for m in PAIR_METRICS:
            a, b = per_seed(first, p, m), per_seed(second, p, m)
            d = a - b
            better = (d < 0) if m == "FP" else (d > 0)
            paired_rows.append({"comparație": name, "primul": first, "al doilea": second, "p": p, "metrică": m,
                                "medie primul": a.mean(), "medie al doilea": b.mean(), "Δ medie (primul − al doilea)": d.mean(),
                                "Δ std": d.std(ddof=1), "Δ min": d.min(), "Δ max": d.max(),
                                "seeds primul mai bun": int(better.sum()), "seeds egale": int((d == 0).sum()),
                                **{f"Δ seed {s}": d[s] for s in SEEDS}})
paired = pd.DataFrame(paired_rows)

# bootstrap stratificat pereche pe tranzacțiile test: doar incertitudine descriptivă, după evaluarea finală
rng_boot = np.random.default_rng(BOOTSTRAP_SEED)
fraud_pos, legit_pos = np.flatnonzero(y_test == 1), np.flatnonzero(y_test == 0)
hist_a_scores = {(s, p): np.load(EXP5_DIR / "scores" / f"A_seed{s}_p{p}.npz")["test"] for s in SEEDS for p in P_FOCUS}
boot = {(name, p): [] for name, *_ in COMPARISONS for p in P_FOCUS}
boot_start = time.perf_counter()
for _ in range(N_BOOTSTRAP):
    idx = np.r_[rng_boot.choice(fraud_pos, len(fraud_pos)), rng_boot.choice(legit_pos, len(legit_pos))]
    yb = y_test[idx]
    lr_mean = np.mean([average_precision_score(yb, TEST_SCORES[LR_KEY, s, 0.0][idx]) for s in SEEDS])
    for p in P_FOCUS:
        means = {o: np.mean([average_precision_score(yb, TEST_SCORES[o, s, p][idx]) for s in SEEDS]) for o in QMODELS}
        means[LR_KEY] = lr_mean
        means[HIST_A] = np.mean([average_precision_score(yb, hist_a_scores[s, p][idx]) for s in SEEDS])
        for name, first, second in COMPARISONS:
            boot[name, p].append(means[first] - means[second])
BOOT_SECONDS = time.perf_counter() - boot_start
boot_rows = []
for (name, p), values in boot.items():
    first, second = next((f, s) for n, f, s in COMPARISONS if n == name)
    observed = per_seed(first, p, "AUPRC").mean() - per_seed(second, p, "AUPRC").mean()
    v = np.asarray(values)
    boot_rows.append({"comparație": name, "p": p, "metrică": "AUPRC (media pe 5 seeds)", "Δ observat": observed,
                      "bootstrap 2.5%": np.percentile(v, 2.5), "bootstrap 97.5%": np.percentile(v, 97.5),
                      "P_bootstrap(Δ > 0)": float((v > 0).mean()), "n bootstrap": N_BOOTSTRAP})
boot_table = pd.DataFrame(boot_rows)
pd.concat([paired, boot_table], ignore_index=True).to_csv(RESULTS_DIR / "paired_test_comparison.csv", index=False)

print("Δ AUPRC pe test (primul − al doilea), medie ± std pe seeds, seeds câștigate de primul:")
display(paired[paired["metrică"] == "AUPRC"].assign(**{"Δ": lambda t: t["Δ medie (primul − al doilea)"].map("{:+.3f}".format) + " ± " + t["Δ std"].map("{:.3f}".format),
                                                        "câștigă primul": lambda t: t["seeds primul mai bun"].astype(str) + "/5"})
        .pivot_table(index="comparație", columns="p", values=["Δ", "câștigă primul"], aggfunc="first"))
print(f"Bootstrap stratificat pereche (descriptiv, {N_BOOTSTRAP} reeșantionări, {BOOT_SECONDS:.0f} s):")
display(boot_table.round(4))

## 9. Validation vs test

In [ ]:
val_rows = []
for o in QMODELS:
    for seed in SEEDS:
        for p in P_ALL:
            r = exp11_row(o, seed, p)
            val_rows.append({"model": o, "seed": seed, "p": p, **{k: float(r[k]) for k in ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP"]}})
for seed in SEEDS:
    m = evaluate(y_val, VAL_SCORES[LR_KEY, seed, 0.0], THRESHOLDS[LR_KEY, seed, 0.0])
    assert abs(m["AUPRC"] - exp4_val[seed]["AUPRC"]) < EXACT_TOL
    for p in P_ALL:
        val_rows.append({"model": LR_KEY, "seed": seed, "p": p, **{k: float(m[k]) for k in ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP"]}})
val_metrics = pd.DataFrame(val_rows)
val_metrics["FP / 10k legitime"] = 1e4 * val_metrics["FP"] / N_LEGIT_VAL


def val_seed(model, p, metric):
    return val_metrics[(val_metrics["model"] == model) & (val_metrics["p"] == p)].set_index("seed").loc[SEEDS, metric]


vt_rows = []
for o in [*QMODELS, LR_KEY]:
    for p in P_ALL:
        for m in ["AUPRC", "AUROC", "recall", "precision", "F1", "MCC", "FP", "FP / 10k legitime", "TP"]:
            v, t = val_seed(o, p, m), per_seed(o, p, m)
            d = t - v
            vt_rows.append({"model": o, "p": p, "metrică": m, "validation medie": v.mean(), "validation std": v.std(ddof=1),
                            "test medie": t.mean(), "test std": t.std(ddof=1), "Δ (test − val) medie": d.mean(), "Δ std": d.std(ddof=1),
                            "seeds test > val": int((d > 0).sum()), **{f"Δ seed {s}": d[s] for s in SEEDS}})
val_vs_test = pd.DataFrame(vt_rows)
for p in P_ALL:   # clasamentul pereche Z0Z1 − Z0 / Z0Z1 − LR se păstrează?
    for other in [QUANTUM_REF, LR_KEY]:
        dv = val_seed(CANDIDATE, p, "AUPRC") - val_seed(other, p, "AUPRC")
        dt = per_seed(CANDIDATE, p, "AUPRC") - per_seed(other, p, "AUPRC")
        vt_rows.append({"model": f"Δ Z0Z1 − {other}", "p": p, "metrică": "AUPRC", "validation medie": dv.mean(), "validation std": dv.std(ddof=1),
                        "test medie": dt.mean(), "test std": dt.std(ddof=1), "Δ (test − val) medie": (dt - dv).mean(), "Δ std": (dt - dv).std(ddof=1),
                        "seeds test > val": int(((dt - dv) > 0).sum()), "seeds Z0Z1 mai bun (val)": int((dv > 0).sum()), "seeds Z0Z1 mai bun (test)": int((dt > 0).sum())})
val_vs_test = pd.DataFrame(vt_rows)
val_vs_test.to_csv(RESULTS_DIR / "validation_vs_test.csv", index=False)
show = val_vs_test[val_vs_test["metrică"].isin(["AUPRC", "recall", "precision", "F1", "FP"])].copy()
show["val"] = show["validation medie"].map("{:.3f}".format) + " ± " + show["validation std"].map("{:.3f}".format)
show["test"] = show["test medie"].map("{:.3f}".format) + " ± " + show["test std"].map("{:.3f}".format)
show["Δ test − val"] = show["Δ (test − val) medie"].map("{:+.3f}".format)
display(show[show["p"].isin([0.0, 0.01, 0.05])].set_index(["model", "metrică", "p"])[["val", "test", "Δ test − val"]])

## 10. Analiza erorilor (doar predicțiile deja obținute)

Pentru fiecare seed comparăm ce fraude detectează Z₀ și Z₀Z₁ și unde dau alarme false. Raportăm numai numere agregate, fără identificatori de tranzacții. Analiza **nu** se folosește pentru a schimba praguri, observabila sau ponderile.

In [ ]:
fraud, legit = y_test == 1, y_test == 0
err_rows = []
for p in P_ALL:
    for seed in SEEDS:
        pred = {o: TEST_SCORES[o, seed, p] >= THRESHOLDS[o, seed, p] for o in [*QMODELS, LR_KEY]}
        zz, z0, lr_ = pred[CANDIDATE], pred[QUANTUM_REF], pred[LR_KEY]
        err_rows.append({"p": p, "seed": seed,
                         "fraude detectate Z0Z1": int((zz & fraud).sum()), "fraude detectate Z0": int((z0 & fraud).sum()),
                         "detectate de ambele": int((zz & z0 & fraud).sum()), "doar Z0Z1": int((zz & ~z0 & fraud).sum()),
                         "doar Z0": int((~zz & z0 & fraud).sum()), "ratate de ambele": int((~zz & ~z0 & fraud).sum()),
                         "fraude detectate LR": int((lr_ & fraud).sum()), "ratate de Z0Z1, Z0 și LR": int((~zz & ~z0 & ~lr_ & fraud).sum()),
                         "FP Z0Z1": int((zz & legit).sum()), "FP Z0": int((z0 & legit).sum()), "FP comune": int((zz & z0 & legit).sum()),
                         "FP doar Z0Z1": int((zz & ~z0 & legit).sum()), "FP doar Z0": int((~zz & z0 & legit).sum()), "FP LR": int((lr_ & legit).sum())})
confusion_analysis = pd.DataFrame(err_rows)
assert (confusion_analysis["detectate de ambele"] + confusion_analysis["doar Z0Z1"] + confusion_analysis["doar Z0"]
        + confusion_analysis["ratate de ambele"] == 98).all()
mean_rows = confusion_analysis.groupby("p").mean(numeric_only=True).drop(columns="seed").assign(seed="medie").reset_index()
confusion_out = pd.concat([confusion_analysis, mean_rows], ignore_index=True)

# stabilitatea între seeds la p = 0.05: în câte seeds este detectată fiecare fraudă (doar numărători)
stab_rows = []
for p in P_FOCUS:
    for o in [*QMODELS, LR_KEY]:
        k = sum((TEST_SCORES[o, s, p] >= THRESHOLDS[o, s, p])[fraud].astype(int) for s in SEEDS)
        stab_rows.append({"p": p, "seed": f"stabilitate {o}", **{f"fraude detectate în {j}/5 seeds": int((k == j).sum()) for j in range(6)}})
confusion_out = pd.concat([confusion_out, pd.DataFrame(stab_rows)], ignore_index=True)
confusion_out.to_csv(RESULTS_DIR / "test_confusion_analysis.csv", index=False)
display(confusion_analysis[confusion_analysis["p"] == 0.05].set_index("seed"))
display(mean_rows.set_index("p").round(1))
display(pd.DataFrame(stab_rows).set_index(["p", "seed"]))

## 11. Costul cuantic

- **Inferența pe test:** o execuție de circuit per tranzacție, per (model, seed, p).
- **Cost nominal pe hardware:** 1024 shots per execuție. **Simulatorul exact nu trage shots.** Execuțiile nominale sunt o estimare pentru hardware, nu execuții efective.
- **Antrenarea** este costul istoric din Exp. 9 / 11 (`result_exp11/cost_ledger.json`) și nu a fost repetată aici.

In [ ]:
exp11_ledger = json.loads((EXP11_DIR / "cost_ledger.json").read_text(encoding="utf-8"))
exp11_shots_ledger = EXP11_DIR / "finite_shot_cost_ledger.json"
n_test = N_TEST_EXPECTED
inference = {}
for o in QMODELS:
    per_level = n_test
    inference[o] = {
        "transactions": n_test, "executions_per_transaction": 1, "cnot_per_execution": STRUCT[o]["CNOT"],
        "noise_locations_per_execution": STRUCT[o]["locuri de zgomot"],
        "executions_per_seed_per_p": per_level, "executions_per_seed_4_p": per_level * len(P_ALL),
        "executions_5_seeds_4_p": per_level * len(P_ALL) * len(SEEDS),
        "nominal_shots_5_seeds_4_p": per_level * len(P_ALL) * len(SEEDS) * SHOTS_PER_EXECUTION,
        "real_device_one_noise_level_executions_5_seeds": per_level * len(SEEDS),
        "real_device_one_noise_level_nominal_shots_5_seeds": per_level * len(SEEDS) * SHOTS_PER_EXECUTION,
        "simulator_seconds_total": float(sum(TEST_TIME[o, s, p] for s in SEEDS for p in P_ALL)),
    }
assert inference[CANDIDATE]["executions_5_seeds_4_p"] == inference[QUANTUM_REF]["executions_5_seeds_4_p"]
training = {o: {k: exp11_ledger["observables"][o][k] for k in
                ["status", "executions_per_example_nominal", "executions_per_example_tracker", "parameters_in_wire_cone",
                 "executions_per_seed", "total_per_seed_nominal", "total_per_seed_tracker", "total_5_seeds_tracker", "shots_5_seeds_tracker"]}
            for o in QMODELS}
ledger = {
    "convention": "exact density-matrix simulation (shots=None): nominal shots are a hardware estimate, NOT drawn by the simulator",
    "shots_per_execution_nominal": SHOTS_PER_EXECUTION,
    "test_inference_exp12": inference,
    "test_inference_same_cost_Z0_vs_Z0Z1": "identical: same circuit (58 gates, 15 CNOT, 73 noise locations), 1 execution per transaction; Z0 and Z0Z1 are read from the same bitstrings, so the shots are not doubled",
    "validation_in_exp12": {"new_protocol_executions": 0,
                            "note": "thresholds reuse the Exp. 11 validation scores (already counted in the Exp. 11 ledger)",
                            "verification_only_simulated_rows": SPOT_CHECK_ROWS * len(QMODELS) * len(SEEDS) * len(NOISE_LEVELS),
                            "exp5_reproduction_check_simulated_rows": n_test * len(SEEDS) + SPOT_CHECK_ROWS * len(SEEDS) * len(NOISE_LEVELS)},
    "training_historical_not_repeated": training,
    "training_gradient_cost_differs": {o: exp11_ledger["observables"][o]["executions_per_example_tracker"] for o in QMODELS},
    "training_tracker_total_Z0Z1_vs_Z0_relative": exp11_ledger["observables"][CANDIDATE]["tracker_total_vs_Z0_relative"],
    "finite_shot_analysis": f"separate protocol (Exp. 11, {exp11_shots_ledger.relative_to(REPO_ROOT).as_posix()}); not part of these exact-expectation metrics",
    "logistic_regression": "classical CPU, 0 quantum executions",
}
(RESULTS_DIR / "test_cost_ledger.json").write_text(json.dumps(to_builtin(ledger), indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame(inference).T)
print("execuții / exemplu în antrenare (Tracker, parameter shift):", ledger["training_gradient_cost_differs"])

## 12. Grafice

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels = ["0", *[str(p) for p in NOISE_LEVELS]]
OFFSETS = {CANDIDATE: -0.12, QUANTUM_REF: 0.0, LR_KEY: 0.12, HIST_A: 0.06}
SUB = "TEST: 56 962 tranzacții, 98 fraude · medie ± std pe 5 seeds (gri: seeds) · prag max-F1 din validation la același p"


def plot_vs_p(metric, ylabel, title, filename, models=(CANDIDATE, QUANTUM_REF, LR_KEY), hist=False):
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    for o in [*models, *([HIST_A] if hist else [])]:
        vals = [per_seed(o, p, metric) for p in P_ALL]
        for i, v in enumerate(vals):
            ax.scatter(np.full(len(v), xpos[i] + OFFSETS[o]), v, s=10, color=SEED_DOT_COLOR, alpha=0.55, zorder=2)
        ax.errorbar(xpos + OFFSETS[o], [v.mean() for v in vals], yerr=[v.std(ddof=1) for v in vals], color=COLORS[o], marker=MARKERS[o],
                    markersize=8, linewidth=2, capsize=3, zorder=3, linestyle="--" if o in (LR_KEY, HIST_A) else "-", label=LABELS[o])
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă (LR: clasic, independent de p)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}\n{SUB}", fontsize=10)
    ax.legend(fontsize=8.5, loc="lower left")
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=130)
    plt.show()


plot_vs_p("AUPRC", "AUPRC (test)", "1 · AUPRC vs p", "01_auprc_vs_p.png", hist=True)
plot_vs_p("recall", "recall (test)", "2 · Recall vs p", "02_recall_vs_p.png")
plot_vs_p("precision", "precision (test)", "3 · Precision vs p", "03_precision_vs_p.png")
plot_vs_p("F1", "F1 (test)", "4 · F1 vs p", "04_f1_vs_p.png")
plot_vs_p("Spearman rho", "Spearman ρ (scoruri cu zgomot vs p = 0)", "5 · Spearman ρ vs p (modele cuantice)", "05_spearman_vs_p.png", models=QMODELS)

In [ ]:
# 6 · AUPRC per seed la p = 0.05
fig, ax = plt.subplots(figsize=(9, 4.6))
order = [CANDIDATE, QUANTUM_REF, LR_KEY]
width = 0.26
for k, o in enumerate(order):
    v = per_seed(o, 0.05, "AUPRC")
    ax.bar(np.arange(len(SEEDS)) + (k - 1) * width, v.to_numpy(), width=width - 0.03, color=COLORS[o], label=f"{LABELS[o]} (medie {v.mean():.3f})")
ax.set_xticks(range(len(SEEDS)), [f"seed {s}" for s in SEEDS])
low = min(per_seed(o, 0.05, "AUPRC").min() for o in order)
ax.set_ylim(max(0, low - 0.08), None)
ax.set_ylabel("AUPRC (test)")
ax.set_title("6 · AUPRC pe test per seed, p = 0.05 (LR: același model la orice p)", fontsize=10)
ax.legend(fontsize=8.5, loc="lower right")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "06_auprc_per_seed_p0.05.png", dpi=130)
plt.show()

# 7 · validation vs test, AUPRC
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for o in order:
    v = [val_seed(o, p, "AUPRC") for p in P_ALL]
    t = [per_seed(o, p, "AUPRC") for p in P_ALL]
    ax.errorbar(xpos + OFFSETS[o], [x.mean() for x in t], yerr=[x.std(ddof=1) for x in t], color=COLORS[o], marker=MARKERS[o],
                markersize=8, linewidth=2, capsize=3, label=f"{SHORT[o]}: test")
    ax.errorbar(xpos + OFFSETS[o] + 0.03, [x.mean() for x in v], yerr=[x.std(ddof=1) for x in v], color=COLORS[o], marker=MARKERS[o],
                markersize=7, linewidth=1.5, capsize=3, linestyle=":", markerfacecolor="white", label=f"{SHORT[o]}: validation")
ax.set_xticks(xpos, xlabels)
ax.set_xlabel("probabilitate depolarizantă p per poartă")
ax.set_ylabel("AUPRC")
ax.set_title("7 · AUPRC: validation (punctat, marker gol) vs test (plin) · medie ± std pe 5 seeds", fontsize=10)
ax.legend(fontsize=8, ncol=3, loc="lower left")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "07_validation_vs_test_auprc.png", dpi=130)
plt.show()

# 8 · Δ AUPRC pereche Z0Z1 − Z0 per seed
fig, ax = plt.subplots(figsize=(8.5, 4.6))
d_all = pd.DataFrame({p: per_seed(CANDIDATE, p, "AUPRC") - per_seed(QUANTUM_REF, p, "AUPRC") for p in P_ALL})
for seed in SEEDS:
    ax.plot(xpos, d_all.loc[seed].to_numpy(), color=SEED_DOT_COLOR, marker="o", markersize=5, linewidth=1, alpha=0.8)
    ax.annotate(f"s{seed}", (xpos[-1], d_all.loc[seed].iloc[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
ax.plot(xpos, d_all.mean().to_numpy(), color=COLORS[CANDIDATE], marker=MARKERS[CANDIDATE], markersize=9, linewidth=2.2, label="medie pe 5 seeds")
for i, p in enumerate(P_ALL):
    ax.annotate(f"{int((d_all[p] > 0).sum())}/5", (xpos[i], d_all[p].max()), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8.5, color="0.25")
ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
ax.set_xticks(xpos, xlabels)
ax.set_xlabel("probabilitate depolarizantă p per poartă")
ax.set_ylabel("Δ AUPRC test (Z₀Z₁ − Z₀)")
ax.set_title("8 · Diferența pereche AUPRC pe test, Z₀Z₁ − Z₀ (gri: seeds; text: seeds cu Z₀Z₁ mai bun)", fontsize=10)
ax.legend(fontsize=8.5, loc="lower left")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "08_paired_delta_auprc_z0z1_minus_z0.png", dpi=130)
plt.show()

In [ ]:
# 9 · matricea de confuzie la p = 0.05, agregată (medie și interval min–max pe 5 seeds)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, o in zip(axes, order):
    sub = test_metrics[(test_metrics["model"] == o) & (test_metrics["p"] == 0.05)]
    cells = [["TN", "FP"], ["FN", "TP"]]
    rate = np.array([[sub["TN"].mean() / 56864, sub["FP"].mean() / 56864], [sub["FN"].mean() / 98, sub["TP"].mean() / 98]])
    ax.imshow(rate, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            c = cells[i][j]
            ax.text(j, i, f"{c}\n{sub[c].mean():,.1f}\n({sub[c].min():,}–{sub[c].max():,})", ha="center", va="center",
                    fontsize=10, color="white" if rate[i, j] > 0.55 else "#1f1f1f")
    ax.set_xticks([0, 1], ["prezis legitim", "prezis fraudă"])
    ax.set_yticks([0, 1], ["legitim\n(56 864)", "fraudă\n(98)"])
    ax.grid(False)
    ax.set_title(LABELS[o], fontsize=9.5)
fig.suptitle("9 · Matrice de confuzie pe test la p = 0.05: medie pe 5 seeds (min–max); culoare = rata pe rând", fontsize=10.5)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "09_confusion_p0.05.png", dpi=130)
plt.show()

# 10 · curbe precision–recall pe test la p = 0.05 (fără prag nou: punctul = pragul din validation)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharey=True)
for ax, o in zip(axes, order):
    for seed in SEEDS:
        s = TEST_SCORES[o, seed, 0.05]
        prec, rec, _ = precision_recall_curve(y_test, s)
        ax.plot(rec, prec, color=COLORS[o], linewidth=1.3, alpha=0.75)
        m = evaluate(y_test, s, THRESHOLDS[o, seed, 0.05])
        ax.scatter(m["recall"], m["precision"], s=36, color=COLORS[o], edgecolor="white", linewidth=1.5, zorder=3)
    ax.axhline(98 / 56962, color="0.5", linestyle=":", linewidth=1)
    ax.set_xlabel("recall")
    ax.set_title(f"{LABELS[o]}\nAUPRC {per_seed(o, 0.05, 'AUPRC').mean():.3f} ± {per_seed(o, 0.05, 'AUPRC').std(ddof=1):.3f}", fontsize=9.5)
axes[0].set_ylabel("precision")
fig.suptitle("10 · Curbe precision–recall pe test, p = 0.05 (o linie per seed; punct = pragul max-F1 din validation, nerecalibrat)", fontsize=10.5)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "10_pr_curves_p0.05.png", dpi=130)
plt.show()

## 13. Verificări și răspunsuri numerice

In [ ]:
check("niciun VQC reantrenat, nicio selecție nouă: Exp. 12 doar încarcă ponderi și aplică praguri înghețate", True)
checks_df = pd.DataFrame(checks)
checks_df.to_csv(RESULTS_DIR / "checks.csv", index=False)
print(f"✅ {len(checks_df)} verificări, toate trecute")
display(checks_df)

answers = {}
for p in P_FOCUS:
    a = per_seed(CANDIDATE, p, "AUPRC")
    answers[f"AUPRC Z0Z1 p = {p}"] = mean_std(a)
    answers[f"AUPRC Z0 p = {p}"] = mean_std(per_seed(QUANTUM_REF, p, "AUPRC"))
    answers[f"seeds Z0Z1 > Z0, p = {p}"] = int((a > per_seed(QUANTUM_REF, p, "AUPRC")).sum())
    answers[f"seeds Z0Z1 > LR, p = {p}"] = int((a > per_seed(LR_KEY, p, "AUPRC")).sum())
    answers[f"TP Z0Z1 p = {p} (din 98)"] = mean_std(per_seed(CANDIDATE, p, "TP"), 1)
    answers[f"FP Z0Z1 p = {p}"] = mean_std(per_seed(CANDIDATE, p, "FP"), 1)
answers["AUPRC LR"] = mean_std(per_seed(LR_KEY, 0.0, "AUPRC"))
answers["AUPRC_drop Z0Z1 (0 → 0.05)"] = mean_std(per_seed(CANDIDATE, 0.05, "AUPRC_drop (vs p = 0)"))
answers["AUPRC_drop Z0 (0 → 0.05)"] = mean_std(per_seed(QUANTUM_REF, 0.05, "AUPRC_drop (vs p = 0)"))
for k, v in answers.items():
    print(f"{k:<32} {v}")

## 14. Concluzii

Interpretarea completă, tabelul final și limitările sunt în `results/result_exp12/report.md`.

> Reamintire: testul a fost folosit anterior în Exp. 5. Rezultatul de aici nu este o validare complet independentă a modelului.